# ✂️ Auto Clipper - Google Colab GPU Backend Server

Notebook ini menjalankan backend **Auto Clipper** di lingkungan GPU Google Colab (T4 GPU Gratis) untuk akselerasi transkripsi Whisper, pemotongan video 9:16 (Face Tracking / Canvas), dan rendering FFmpeg NVENC.

---

### 1. Hubungkan Google Drive (Mount Drive)
Hubungkan Google Drive Anda untuk menyimpan riwayat proyek, video asli, database (`history.db`), dan klip hasil render secara permanen.

In [1]:
import os
try:
  import torch
  if not torch.cuda.is_available():
    print("\n=======================================================")
    print("❌ PERINGATAN KRITIS: GPU TIDAK AKTIF / ANDA MEMAKAI CPU!")
    print("\n👉 Pantesan prosesnya lemot banget! Google Colab Anda kehabisan kuota GPU atau salah pengaturan.")
    print("👉 CARA FIX: Klik menu atas \"Runtime\" -> \"Change runtime type\" -> Pilih Hardware accelerator: \"T4 GPU\" lalu Save.")
    print("=======================================================\n")
  else:
    print("✅ GPU T4 Aktif dan siap digunakan!\n")
except Exception:
  pass

from google.colab import drive
drive.mount('/content/drive')

✅ GPU T4 Aktif dan siap digunakan!

Mounted at /content/drive


### 2. Pasang Sistem & Dependensi (FFmpeg & Cloudflared)
Memasang `ffmpeg` untuk pengolah video dan `cloudflared` untuk tunnel jaringan aman.

In [2]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package cloudflared.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.11.1) ...
Setting up cloudflared (2026.11.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


### 3. Unduh Program Auto Clipper & Dependensi Python
Mengunduh repositori dan memasang seluruh library Python yang dibutuhkan.

In [5]:
!git clone https://github.com/Maulidan45/autoclip.git /content/auto-clipper || (cd /content/auto-clipper && git pull)
%cd /content/auto-clipper
!pip install -r backend/requirements.txt uvicorn pyngrok
!pip install -U yt-dlp

Cloning into '/content/auto-clipper'...
remote: Enumerating objects: 3645, done.
remote: Counting objects: 100% (3645/3645), done.
remote: Compressing objects: 100% (1053/1053), done.
remote: Total 3645 (delta 2480), reused 3645 (delta 2480), pack-reused 0 (from 0)
Receiving objects: 100% (3645/3645), 12.47 MiB | 19.00 MiB/s, done.
Resolving deltas: 100% (2480/2480), done.
/content/auto-clipper
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 73.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.3/28.3 MB 154.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 51.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 92.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 MB 68.3 MB/s  0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-head

### 4A. Jalankan Backend (Pilihan A: Cloudflare Tunnel - Punya Domain Sendiri)
Masukkan **Cloudflare Tunnel Token** dan **API Secret Token (Password Rahasia)** pada formulir di bawah ini, lalu klik tombol **Play (▶️)**.

In [ ]:
%cd /content/auto-clipper
#@title Jalankan Server via Cloudflare Tunnel
CLOUDFLARE_TUNNEL_TOKEN = "" #@param {type:"string"}
API_SECRET_TOKEN = "" #@param {type:"string"}

!python backend/colab_api.py --cloudflare-token "$CLOUDFLARE_TUNNEL_TOKEN" --api-token "$API_SECRET_TOKEN"

### 4B. Jalankan Backend (Pilihan B: Ngrok Tunnel - 100% Gratis Tanpa Domain)
Jika Anda tidak memiliki domain pribadi di Cloudflare, gunakan formulir di bawah ini dengan **Ngrok Authtoken** (Daftar gratis di ngrok.com).

In [ ]:
%cd /content/auto-clipper
#@title Jalankan Server via Ngrok Tunnel
NGROK_AUTHTOKEN = "3MSYIIVeYxkAb9UEttYT9ZfakV2_4HkEqxqbaLHFTPfHtsbHh" #@param {type:"string"}
API_SECRET_TOKEN = "9f4c2a7d1b8e6c3f0a5d9e2b7c1f4a8d" #@param {type:"string"}

from pyngrok import ngrok
import os

os.environ["AUTO_CLIPPER_CORS_ORIGIN"] = "https://autoclip-production-256eab1c.up.railway.app"

if NGROK_AUTHTOKEN.strip():
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())

public_url = ngrok.connect(8000).public_url
print(f"\n=========================================")
print(f"🚀 PUBLIC BACKEND URL ANDA: {public_url}")
print(f"=========================================\n")

os.environ["API_SECRET_TOKEN"] = API_SECRET_TOKEN.strip()
os.environ["AUTO_CLIPPER_DEV_TOKEN"] = API_SECRET_TOKEN.strip()
os.environ["AUTO_CLIPPER_WEB_TOKEN"] = API_SECRET_TOKEN.strip()
os.environ["AUTO_CLIPPER_CLOUD_MODE"] = "1"
os.environ["AUTO_CLIPPER_WORKSPACE"] = "/content/drive/MyDrive/AutoClipperData"

try:
    import nvidia.cublas.lib
    import nvidia.cudnn.lib
    os.environ["LD_LIBRARY_PATH"] = os.path.dirname(nvidia.cublas.lib.__file__) + ":" + os.path.dirname(nvidia.cudnn.lib.__file__) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
except Exception:
    os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

!python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000


/content/auto-clipper

🚀 PUBLIC BACKEND URL ANDA: https://carolina-sandlike-votively.ngrok-free.dev

INFO:     Started server process [20648]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     2402:5680:88ca:6f79:f9f1:6c42:83d1:7bef:0 - "OPTIONS /api/settings/whisper-models HTTP/1.1" 200 OK
INFO:     2402:5680:88ca:6f79:f9f1:6c42:83d1:7bef:0 - "GET / HTTP/1.1" 401 Unauthorized
INFO:     2402:5680:88ca:6f79:f9f1:6c42:83d1:7bef:0 - "GET /favicon.ico HTTP/1.1" 401 Unauthorized
INFO:     2402:5680:88ca:6f79:f9f1:6c42:83d1:7bef:0 - "GET /health HTTP/1.1" 200 OK
INFO:     2402:5680:88ca:6f79:f9f1:6c42:83d1:7bef:0 - "GET /favicon.ico HTTP/1.1" 401 Unauthorized
INFO:     2402:5680:88ca:6f79:f9f1:6c42:83d1:7bef:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     2402:5680:88ca:6f79:f9f1:6c42:83d1:7bef:0 - "OPTIONS /api/settings/whisper-models HTTP/1.1" 200 OK
INFO:     2402:5680:88ca:6f